# Build a 2,000-image three-branch meta-dataset

This notebook draws a reproducible, diverse, balanced sample of **1,000 authentic (0)** and **1,000 AI-generated (1)** images from the held-out validation manifest. Sampling rotates across source/generator groups so large groups cannot dominate. Each image is scored by:

1. Community Forensics,
2. the trained DINOv3-L detector, and
3. the trained DINOv3-B stacking detector.

The fourth main column is the **known ground-truth label**, not a prediction. A ground-truth target is required to learn and evaluate a new final-fusion function. The stacking score is not independent: its learned head already consumes the Community Forensics logit.

> Run this notebook from the project environment. The first Community Forensics or DINOv3-B run may use the Hugging Face cache or download missing released weights.

In [1]:
from pathlib import Path
import gc
import json
import os
import sys

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'Modeling').exists():
    candidate = PROJECT_ROOT / 'Coding'
    if candidate.exists():
        PROJECT_ROOT = candidate
    else:
        PROJECT_ROOT = Path.cwd().resolve().parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault('HF_HOME', str(PROJECT_ROOT / 'cache' / 'hf'))

from Data_Pipeline.image_io import load_image, resolve_path
from Modeling.registry import get

SEED = 1337
N_PER_CLASS = 1000
# This planned validation subset is the row-aligned source of the existing
# Community Forensics and DINOv3-B caches (12,000 labeled images).
MANIFEST = PROJECT_ROOT / 'data/plan/validation.csv'
DINO_L_CHECKPOINT = PROJECT_ROOT / 'weights/dinov3/notebook_dinov3/best_robust.pt'
STACKING_CHECKPOINT = PROJECT_ROOT / 'weights/stacking/notebook_vitb/best_robust.pt'
OUTPUT_DIR = PROJECT_ROOT / 'outputs/meta_fusion'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAIN_CSV = OUTPUT_DIR / 'meta_training_2000.csv'
AUDIT_CSV = OUTPUT_DIR / 'meta_training_2000_audit.csv'

print('Project:', PROJECT_ROOT)
print('Torch device availability:', {'cuda': torch.cuda.is_available(), 'mps': torch.backends.mps.is_available()})

Project: /Users/dunglai/Desktop/Việt Dũng/Hackathon/Tiktok/Coding
Torch device availability: {'cuda': False, 'mps': True}


## 1. Select a balanced, reproducible sample

Sampling from validation rather than the original detector-training manifest reduces leakage. The source path and hashes are retained in a separate audit file, while the requested training CSV contains exactly four columns.

In [3]:
manifest = pd.read_csv(MANIFEST)
manifest = manifest.reset_index(names='manifest_row')
group_columns = [column for column in ['source', 'generator'] if column in manifest.columns]
manifest['_diversity_group'] = (
    manifest[group_columns].fillna('unknown').astype(str).agg(' / '.join, axis=1)
    if group_columns else 'all'
)

def diverse_sample(pool, n, seed):
    # Round-robin through independently shuffled groups. This retains rare
    # generators while remaining random and reproducible within each group.
    rng = np.random.default_rng(seed)
    buckets = {}
    for group, rows in pool.groupby('_diversity_group', sort=True):
        indices = rows.index.to_numpy(copy=True)
        rng.shuffle(indices)
        buckets[group] = indices.tolist()
    groups = list(buckets)
    rng.shuffle(groups)
    selected = []
    while len(selected) < n:
        for group in groups:
            if buckets[group]:
                selected.append(buckets[group].pop())
                if len(selected) == n:
                    break
    return pool.loc[selected].copy()

parts = []
for label in (0, 1):
    pool = manifest.loc[manifest['label'] == label].copy()
    parts.append(diverse_sample(pool, N_PER_CLASS, SEED + label))

sample = (
    pd.concat(parts, ignore_index=True)
    .sample(frac=1.0, random_state=SEED)
    .reset_index(drop=True)
)
sample['resolved_path'] = sample['path'].map(lambda p: str(resolve_path(p)))

print(sample['label'].value_counts().sort_index().rename(index={0: 'authentic', 1: 'AI-generated'}))
print('Diversity groups represented:', sample['_diversity_group'].nunique())
display(pd.crosstab(sample['_diversity_group'], sample['label']).sort_values([0, 1], ascending=False).head(20))
display(sample[['path', 'label', 'source', 'generator']].head(10))

label
authentic       1000
AI-generated    1000
Name: count, dtype: int64
Diversity groups represented: 28


label,0,1
_diversity_group,,
sid_set / real,143,0
wildfake / afhq,143,0
wildfake / church,143,0
wildfake / coco,143,0
wildfake / ffhq,143,0
wildfake / imagenet,143,0
wildfake / celebahq,142,0
sid_set / full_synthetic,0,48
wildfake / BigGAN,0,48


,path,label,source,generator
0,data/evaluation/images/validation/sid_set/real...,0,sid_set,real
1,data/evaluation/images/validation/wildfake/rea...,0,wildfake,imagenet
2,data/evaluation/images/validation/wildfake/rea...,0,wildfake,church
3,data/evaluation/images/validation/wildfake/rea...,0,wildfake,imagenet
4,data/evaluation/images/validation/wildfake/fak...,1,wildfake,originalSD_Advanced
5,data/evaluation/images/validation/wildfake/rea...,0,wildfake,coco
6,data/evaluation/images/validation/wildfake/fak...,1,wildfake,starGAN
7,data/evaluation/images/validation/wildfake/rea...,0,wildfake,church
8,data/evaluation/images/validation/sid_set/full...,1,sid_set,full_synthetic
9,data/evaluation/images/validation/wildfake/fak...,1,wildfake,starGAN


## 2. Score the three branches

Models are constructed and released one at a time. This is slower than keeping all three resident, but it avoids holding both large DINO backbones and two Community Forensics instances in memory simultaneously.

In [4]:
def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


def score_in_batches(model_name, model_kwargs, paths, batch_size, description, cache_name=None):
    cache_path = OUTPUT_DIR / f'{cache_name}.npy' if cache_name else None
    done_path = OUTPUT_DIR / f'{cache_name}.done.npy' if cache_name else None
    if cache_path and cache_path.exists() and done_path.exists():
        output = np.load(cache_path, mmap_mode='r+')
        done = np.load(done_path)
    elif cache_path:
        output = np.lib.format.open_memmap(cache_path, mode='w+', dtype=np.float64, shape=(len(paths),))
        done = np.zeros(len(paths), dtype=bool)
        np.save(done_path, done)
    else:
        output = np.empty(len(paths), dtype=np.float64)
        done = np.zeros(len(paths), dtype=bool)
    if done.all():
        print(f'{description}: reused complete resumable cache')
        return np.asarray(output).copy()
    scorer = get(model_name, **model_kwargs)
    print(scorer.describe() if hasattr(scorer, 'describe') else f'{model_name}: {scorer.n_params():,} parameters')
    try:
        for start in tqdm(range(0, len(paths), batch_size), desc=description):
            stop = min(start + batch_size, len(paths))
            indices = np.arange(start, stop)[~done[start:stop]]
            if not len(indices):
                continue
            images = [load_image(paths[index]) for index in indices]
            output[indices] = scorer.score_checked(images)
            done[indices] = True
            if cache_path:
                output.flush()
                np.save(done_path, done)
            for image in images:
                image.close()
    finally:
        del scorer
        release_memory()
    return np.asarray(output).copy()


paths = sample['resolved_path'].tolist()
manifest_rows = sample['manifest_row'].to_numpy(dtype=int)
scores = {}

# Reuse the full validation cache when available.
community_full = None
try:
    from Workflows.cache_scores import load_scores
    cached, done = load_scores(
        MANIFEST, model='community_forensics', variant='clean',
        root=PROJECT_ROOT / 'cache' / 'scores',
    )
    community_full = cached
    scores['Community Forensics Pr'] = cached[manifest_rows].astype(np.float64)
    print('Community Forensics: reused validation score cache')
except Exception as error:
    print(f'Community Forensics cache unavailable ({error}); running inference')
    scores['Community Forensics Pr'] = score_in_batches(
        'community_forensics', {}, paths, batch_size=8, description='Community Forensics'
    )

# DINOv3-L has no full validation cache, so this remains the expensive branch.
scores['DINOv3-L detector Pr'] = score_in_batches(
    'dinov3', {'checkpoint': str(DINO_L_CHECKPOINT), 'n_crops': 8},
    paths, batch_size=2, description='DINOv3-L',
    cache_name=f'dinov3_l_seed{SEED}_{2 * N_PER_CLASS}',
)

# The stacking head can run directly on the already-cached DINOv3-B features.
try:
    from Modeling.stacking.features import FeatureSet
    from Modeling.stacking.head import load as load_stacking_head, predict_variant
    head, state = load_stacking_head(STACKING_CHECKPOINT, map_location='cpu')
    full_features = FeatureSet(
        MANIFEST, backbone=state['backbone'], crops=int(state['crops']),
        crop_size=int(state['crop_size']), variants=['clean'],
        root=PROJECT_ROOT / 'cache' / 'features', extra=community_full,
    )
    position_by_manifest_row = {int(row): pos for pos, row in enumerate(full_features.index)}
    positions = np.array([position_by_manifest_row[int(row)] for row in manifest_rows])
    selected_features = full_features.select(positions)
    scores['DINOv3-B stacking detector Pr'] = predict_variant(
        head, selected_features, 'clean', device=torch.device('cpu')
    )
    del head, full_features, selected_features
    release_memory()
    print('DINOv3-B stacking: reused validation feature cache')
except Exception as error:
    print(f'DINOv3-B feature cache unavailable ({error}); running inference')
    scores['DINOv3-B stacking detector Pr'] = score_in_batches(
        'stacking', {'checkpoint': str(STACKING_CHECKPOINT)},
        paths, batch_size=4, description='DINOv3-B stacking'
    )

Community Forensics: reused validation score cache


dinov3: 313,057,281 parameters


DINOv3-L:   0%|          | 0/1000 [00:00<?, ?it/s]

DINOv3-B stacking: reused validation feature cache


## 3. Build and save the requested dataframe

`Real/Fake image (0/1)` is the target: **0 = authentic, 1 = AI-generated**. The three probability columns are model inputs for the next notebook.

In [5]:
result = pd.DataFrame(scores)
result['Real/Fake image (0/1)'] = sample['label'].astype(int).to_numpy()

probability_columns = [
    'Community Forensics Pr',
    'DINOv3-L detector Pr',
    'DINOv3-B stacking detector Pr',
]

result.to_csv(MAIN_CSV, index=False)
audit_columns = [c for c in ['manifest_row', 'path', 'resolved_path', 'label', 'source', 'generator', '_diversity_group', 'sha256', 'width', 'height'] if c in sample]
audit = pd.concat([sample[audit_columns].reset_index(drop=True), result], axis=1)
audit.to_csv(AUDIT_CSV, index=False)

print('Saved:', MAIN_CSV)
print('Saved:', AUDIT_CSV)
display(result.head(10))
display(result.describe().T)

Saved: /Users/dunglai/Desktop/Việt Dũng/Hackathon/Tiktok/Coding/outputs/meta_fusion/meta_training_2000.csv
Saved: /Users/dunglai/Desktop/Việt Dũng/Hackathon/Tiktok/Coding/outputs/meta_fusion/meta_training_2000_audit.csv


,Community Forensics Pr,DINOv3-L detector Pr,DINOv3-B stacking detector Pr,Real/Fake image (0/1)
0,0.000065,0.210331,0.213508,0
1,0.000001,0.010722,0.155117,0
2,0.000002,0.004868,0.135821,0
3,0.000015,0.018738,0.123739,0
4,0.940172,0.891083,0.934058,1
5,0.000009,0.009181,0.170754,0
6,0.501921,0.963829,0.463077,1
7,0.000001,0.064210,0.164555,0
8,0.999892,0.971347,0.924103,1
9,0.003278,0.979826,0.423359,1


,count,mean,std,min,25%,50%,75%,max
Community Forensics Pr,2000.0,0.432637,0.477011,1.886278e-07,0.000050,0.021443,0.999856,1.000000
DINOv3-L detector Pr,2000.0,0.499790,0.440186,2.685698e-03,0.025742,0.502822,0.964521,0.991138
DINOv3-B stacking detector Pr,2000.0,0.540635,0.349412,4.772643e-02,0.185868,0.423659,0.920337,0.999766
Real/Fake image (0/1),2000.0,0.500000,0.500125,0.000000e+00,0.000000,0.500000,1.000000,1.000000


In [6]:
from sklearn.metrics import accuracy_score, roc_auc_score

y = result['Real/Fake image (0/1)'].to_numpy()
baseline_rows = []
for column in probability_columns:
    p = result[column].to_numpy()
    baseline_rows.append({
        'branch': column,
        'accuracy_at_0.5': accuracy_score(y, p >= 0.5),
        'roc_auc': roc_auc_score(y, p),
    })
baseline_metrics = pd.DataFrame(baseline_rows).sort_values('accuracy_at_0.5', ascending=False)
display(baseline_metrics.style.format({'accuracy_at_0.5': '{:.2%}', 'roc_auc': '{:.4f}'}))

summary = {
    'seed': SEED,
    'rows': len(result),
    'authentic_rows': int((y == 0).sum()),
    'ai_generated_rows': int((y == 1).sum()),
    'source_manifest': str(MANIFEST),
    'main_csv': str(MAIN_CSV),
    'audit_csv': str(AUDIT_CSV),
    'diversity_groups': int(sample['_diversity_group'].nunique()),
}
(OUTPUT_DIR / 'meta_training_2000_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
summary

,branch,accuracy_at_0.5,roc_auc
1,DINOv3-L detector Pr,98.75%,0.9990
2,DINOv3-B stacking detector Pr,95.65%,0.9911
0,Community Forensics Pr,91.25%,0.9788


{'seed': 1337,
 'rows': 2000,
 'authentic_rows': 1000,
 'ai_generated_rows': 1000,
 'source_manifest': '/Users/dunglai/Desktop/Việt Dũng/Hackathon/Tiktok/Coding/data/plan/validation.csv',
 'main_csv': '/Users/dunglai/Desktop/Việt Dũng/Hackathon/Tiktok/Coding/outputs/meta_fusion/meta_training_2000.csv',
 'audit_csv': '/Users/dunglai/Desktop/Việt Dũng/Hackathon/Tiktok/Coding/outputs/meta_fusion/meta_training_2000_audit.csv',
 'diversity_groups': 28}